# 电路(circuit)约束：用一条回路把所有顶点串起来

## 这一章讲什么

CP-SAT 里的 `add_circuit()` 只管一件事，即把所有顶点各经过 1 次串成一个回路，凡是路怎么走、顺序怎么排这类问题，交给它一个就够。这一章把这件工具用熟，下面先列出这一章的要点。

- `add_circuit()` 怎么用：给一张边的列表，每条边配一个布尔变量
- 有些顶点可以不走，用自环 `(i, i)` 表达
- 指定某条边必过、只砍掉几个顶点、求 `src → dst` 的路径
- 最后拿骑士巡游和格雷码练手，把它们当回路来解

## `add_circuit()` 是什么

有些问题问的不是挑哪几个，而是按什么顺序走，例如 DNA 的片段按什么次序拼起来、一轮实验怎么排先后，都属于这一类。

`add_circuit()` 收下一串 `(起点, 终点, 布尔变量)`，保证只挑那些变量为真的边时，走出来的是一条闭路，每个顶点恰好经过 1 次。

用它之前有两个前提需要注意。

- 顶点得从 0 开始连续编号，中间缺一号，回路就拼不起来
- 想让某个顶点待在回路外面，就给它加一条 `(i, i, var)` 自环

道理先放一放，下面拿这张无向图试一下，看过一遍就很清楚了。

```{mermaid}
graph
0 <--> 1
0 <--> 2
0 <--> 3
1 --> 2
1 <--> 3
2 <--> 3
```

In [1]:
from ortools.sat.python import cp_model
from helper.ortools import get_all_solutions
import numpy as np

# 把无向图变成两个方向的有向边
edges = [(0, 1), (0, 2), (0, 3), (1, 0), (1, 2), (1, 3),
         (2, 0), (2, 3), (3, 0), (3, 1), (3, 2)]


def create_circuit_model(edges, include_edges=[]):
    """由边的列表，建立寻找回路的模型"""
    edges = set(edges)
    model = cp_model.CpModel()
    if include_edges:
        for edge in include_edges:
            if edge not in edges:
                edges.add(edge)

    variables = {key: model.new_bool_var(str(key)) for key in edges}
    if include_edges:
        # 必须经过的边固定为 True
        model.add_bool_and([variables[edge] for edge in include_edges])

    model.add_circuit([(s, e, v) for (s, e), v in variables.items()])
    return model, variables

## 求一个回路

`solver.solve()` 一跑，再把每条边那个布尔变量的值取出来，回路就在里面；取出来的边都是变量为真的边，下面先看代码。

In [2]:
model, variables = create_circuit_model(edges)
solver = cp_model.CpSolver()
solver.solve(model)
[(s, e, solver.value(v)) for (s, e), v in variables.items()]

[(0, 1, 0),
 (1, 2, 0),
 (3, 1, 1),
 (0, 3, 0),
 (2, 0, 0),
 (3, 0, 0),
 (2, 3, 1),
 (0, 2, 1),
 (1, 0, 1),
 (3, 2, 0),
 (1, 3, 0)]

只看值为 `True` 的边，把它们画出来应该就是一圈。可是这么看太费劲，因此下面写一个函数，把它换成顶点的序列再输出。

In [3]:
def get_circuit(solution, start=None):
    """从为真的边取出顶点的顺序"""
    # 去掉自环（不经过的顶点）
    solution = {s: e for (s, e), v in solution.items() if v and s != e}
    if start is None:
        start = next(iter(solution.keys()))
    path = [start]
    while True:
        if start not in solution:
            break
        start = solution[start]
        if start in path:
            break
        path.append(start)
    return path


def find_all_circuits(edges, include_edges=[], start=None):
    """求出所有回路"""
    model, variables = create_circuit_model(edges, include_edges)
    solutions = get_all_solutions(model, variables)
    return [get_circuit(sol, start) for sol in solutions]


def find_one_circuit(edges, include_edges=[], start=None):
    """求出一个回路"""
    model, variables = create_circuit_model(edges, include_edges)
    solver = cp_model.CpSolver()
    solver.solve(model)
    sol = {key: solver.value(v) for key, v in variables.items()}
    return get_circuit(sol, start)

下面把所有的回路都显示出来，逐个核对其中边的顺序是否绕成一圈。

In [4]:
for circuit in find_all_circuits(edges):
    print(circuit)

[3, 1, 0, 2]
[1, 2, 0, 3]
[0, 1, 3, 2]
[0, 1, 2, 3]


## 必须经过指定的边

把边交给 `include_edges`，它就替我们把这些边钉成 `True`。下面这个例子要求回路一定走过边 `(2, 1)`，其余边照旧由求解器挑选。

In [5]:
for circuit in find_all_circuits(edges, include_edges=[(2, 1)]):
    print(circuit)

[2, 1, 3, 0]
[2, 1, 0, 3]


## 允许不经过某些顶点

想放掉某个顶点，就在边的列表里给它加一条 `(n, n)` 这样的自环。下面这个例子允许把顶点 2 和顶点 3 一起省掉。

In [6]:
for circuit in find_all_circuits(edges + [(2, 2), (3, 3)]):
    print(circuit)

[3, 1, 0]
[3, 1, 0, 2]
[1, 2, 0, 3]
[0, 1, 3]
[0, 1, 3, 2]
[0, 1]
[0, 1, 2]
[0, 1, 2, 3]


## 寻找从 `src` 到 `dst` 的路径

从 `src` 到 `dst` 的路径可以反过来看：求一条回路，让它一定走过 `dst → src` 这条边。路上不想经过的顶点，提前把它的自环加进边的列表里即可。

In [7]:
def find_all_paths(edges, src, dst):
    return find_all_circuits(edges, include_edges=[(dst, src)], start=src)


for path in find_all_paths(edges + [(1, 1), (2, 2)], 0, 3):
    print(path)

[0, 2, 3]
[0, 3]
[0, 1, 3]
[0, 1, 2, 3]


(ortools-knight)=
## 例：骑士巡游问题

[骑士巡游问题](z3-knight)要求每个格子都访问 1 次，这本身就是一条回路。因此把骑士一步能跳到的格子之间连上边，回路模型就搭好了，剩下的交给求解器。

In [8]:
n = 8
moves = [(1, 2), (2, 1), (-1, 2), (2, -1), (-2, 1), (1, -2), (-1, -2), (-2, -1)]
board = np.arange(0, n ** 2).reshape(n, n)

# 把骑士一步能走到的格子连成边
edges = []
for (r, c), v in np.ndenumerate(board):
    for dr, dc in moves:
        r2, c2 = r + dr, c + dc
        if 0 <= r2 < n and 0 <= c2 < n:
            edges.append((int(v), int(board[r2, c2])))

steps = find_one_circuit(edges)

# 把第几个访问写到棋盘上
solution = np.zeros((n, n), np.uint8)
for i, step in enumerate(steps):
    r, c = step // n, step % n
    solution[r, c] = i
print(solution)

[[16 11 40 45 48 63 22 43]
 [39 46 17 62 41 44 49  0]
 [12 15 10 47 18 21 42 23]
 [ 5 38 61 14 53 50  1 20]
 [60 13  4  9  2 19 24 51]
 [37  6 35 54 57 52 27 30]
 [34 59  8  3 32 29 56 25]
 [ 7 36 33 58 55 26 31 28]]


(ortools-gray-code)=
## 例：格雷码

[格雷码](z3-gray-code)问的是怎样把只差 1 位的数排成一列，本质上还是一条回路。把只差 1 位的整数两两连成边，题目就变成了回路问题，做法与前一个例子相同。

In [9]:
n = 4
bits = {2 ** i for i in range(n)}

edges = []
for i in range(2 ** n):
    for j in range(2 ** n):
        if i != j and (i ^ j) in bits:
            edges.append((i, j))

codes = find_one_circuit(edges, start=0)
for code in codes:
    print(f"{code:0{n}b}")

0000
1000
1100
1101
1001
1011
1111
1110
1010
0010
0110
0111
0011
0001
0101
0100


## 小结

| 想做的事 | 怎么写 |
|---|---|
| 经过所有顶点的回路 | `add_circuit([(u, v, var), ...])` |
| 必须经过指定的边 | 用 `add_bool_and` 把这条边的变量钉在 `True`(`include_edges`) |
| 允许不经过某些顶点 | 再加上自环 `(i, i, var)` |
| 从 `src` 到 `dst` 的路径 | 改写成“必过 `(dst, src)`”的回路 |
| 取出巡回的顺序 | 顺着为真的边走(`get_circuit()`) |

- 顶点从 0 起连续编号，缺一个号就搭不出回路
- 自环的含义很直接：这个顶点这一次不进入回路
- 碰上“依次各走 1 次”的题目，例如骑士巡游和格雷码，都能翻成回路